In [10]:
import argparse
import time
import numpy as np
import tkinter as tk
from tkinter import filedialog
from PIL import Image, ImageTk
from brainflow.board_shim import BoardShim, BrainFlowInputParams, BoardIds, BrainFlowPresets
import pandas as pd
import matplotlib.pyplot as plt
from brainflow.data_filter import DataFilter, FilterTypes, AggOperations, NoiseTypes, WaveletTypes, NoiseEstimationLevelTypes, \
WaveletExtensionTypes, ThresholdTypes, WaveletDenoisingTypes, WindowOperations, DetrendOperations
import os
import sys
import ast
import scipy
from scipy import stats
from scipy import signal
from scipy.signal import spectrogram
import mne.io
from mne.io import read_raw_edf
import glob

In [24]:
root = tk.Tk()
root.attributes("-topmost", True) #Open window on top of other running applications
root.withdraw()
folder_path = filedialog.askdirectory()
if not os.path.exists(folder_path):
    os.makedirs(folder_path)
session_name = os.path.split(folder_path)[1]
files=[]
run_folders=[]
metadata_df= pd.DataFrame()
for f in os.listdir(folder_path):
    name, ext = os.path.splitext(f)
    if ext == '.edf':
        files.append(f)

FileNotFoundError: [Errno 2] No such file or directory: ''

In [22]:
#To see channels data use print(raw.info)
phantom_chs=['R Frontal arrow up','R Frontal arrow down','R Medial','R Posterial','L Frontal arrow up','L Frontal arrow down','L Medial','L Posterial']

for j in range (len(files)):
    # Load EDF file with filterd data
    raw=mne.io.read_raw_edf(os.path.join(folder_path,files[j]),preload=True)
    fs=int(raw.info['sfreq']) #Get sampling frequency
    datas, times = raw[:,:] #raw in Volts
    channel_names=raw.info["ch_names"]

    #Trim data by time line which is in seconds
    start=90 #90
    end=150 #105(15s), 120(30s), 150(60s)
    time_mask=(times >= start) & (times<=end)
    datas=datas[:,time_mask]
    times=times[time_mask]
    times=times*1000000 #turn seconds to microseconds
    datas=datas*1000000 #turn Volts to microVolts but note that STG4002 will read it as mV
    
    chosen_channels=[]
    for ch_name in phantom_chs:
        chose_channel=input(f'Showing channel list {channel_names} for {files[0]}. \nChoose EEG channel for phantom channel by printing their order number. Press enter for empty phantom channel \n{ch_name}:')
        chosen_channels.append(chose_channel)

    multiplication_factor=1 #multiply for larger stimuli magnitude
    #Avoid overwriting files
    filename = f'EEGx{multiplication_factor}.dat'
    base, ext = os.path.splitext(filename)
    counter = 1
    new_filename = filename
    while os.path.exists(os.path.join(folder_path,new_filename)):
        new_filename = f"{base}_{counter}{ext}"
        counter += 1

    f= open(os.path.join(folder_path,new_filename),'wb+')
    head='Multi Channel Systems MC_Stimulus \nASCII import Version 1.10 \nchannels: 8 \noutput mode: voltage \nformat: 	4 \n '
    f.write(head.encode('ascii'))
    eeg_data_ascii=pd.DataFrame([],columns=['1','Time'])
    for i in range (len(chosen_channels)):
        if chosen_channels[i]=='':
            eeg_data_ascii['1']=datas[0][:-1]*0 #Null empty phantom channels
            eeg_data_ascii['Time']=np.diff(times)
            eeg_data_ascii=eeg_data_ascii[['1', 'Time']]
            channel_number='\nchannel: '+str(i+1)+'\nvalue	time \n'
            f.write(channel_number.encode('ascii'))
            eeg_data_ascii.to_csv(f, sep='\t', index=False, header=False,lineterminator='\n')
        else:
            eeg_data_ascii['1']=datas[int(chosen_channels[i])-1][:-1]*multiplication_factor #chosen_channels-1 since Python counts from 0
            eeg_data_ascii['Time']=np.diff(times)
            eeg_data_ascii=eeg_data_ascii[['1', 'Time']]
            channel_number='\nchannel: '+str(i+1)+'\nvalue	time \n'
            f.write(channel_number.encode('ascii'))
            eeg_data_ascii.to_csv(f, sep='\t', index=False, header=False,lineterminator='\n')
    f.close()

Extracting EDF parameters from /Users/lucanleung/Desktop/EDF_ASCII/Subject 04/1/Subject04_1.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 84999  =      0.000 ...   169.998 secs...


Showing channel list ['EEG Fp1', 'EEG Fp2', 'EEG F3', 'EEG F4', 'EEG F7', 'EEG F8', 'EEG T3', 'EEG T4', 'EEG C3', 'EEG C4', 'EEG T5', 'EEG T6', 'EEG P3', 'EEG P4', 'EEG O1', 'EEG O2', 'EEG Fz', 'EEG Cz', 'EEG Pz', 'EEG A2-A1', 'ECG ECG'] for Subject04_1.edf. 
Choose EEG channel for phantom channel by printing their order number. Press enter for empty phantom channel 
R Frontal arrow up: 4
Showing channel list ['EEG Fp1', 'EEG Fp2', 'EEG F3', 'EEG F4', 'EEG F7', 'EEG F8', 'EEG T3', 'EEG T4', 'EEG C3', 'EEG C4', 'EEG T5', 'EEG T6', 'EEG P3', 'EEG P4', 'EEG O1', 'EEG O2', 'EEG Fz', 'EEG Cz', 'EEG Pz', 'EEG A2-A1', 'ECG ECG'] for Subject04_1.edf. 
Choose EEG channel for phantom channel by printing their order number. Press enter for empty phantom channel 
R Frontal arrow down: 2
Showing channel list ['EEG Fp1', 'EEG Fp2', 'EEG F3', 'EEG F4', 'EEG F7', 'EEG F8', 'EEG T3', 'EEG T4', 'EEG C3', 'EEG C4', 'EEG T5', 'EEG T6', 'EEG P3', 'EEG P4', 'EEG O1', 'EEG O2', 'EEG Fz', 'EEG Cz', 'EEG Pz', 